In [1]:
import pandas as pd 

In [2]:
df = pd.read_parquet('processed\\df_staging_2.parquet')

In [3]:
df_x = df.drop(df.columns[403:],axis=1)
df_y = df.drop(df.columns[:403],axis=1)

In [4]:
df_x = df_x.drop(columns=['supported_languages','full_audio_languages'])
labels = list(df_y.columns)


In [5]:
genres_cols = [c for c in df_y.columns if c.startswith("genres:")]
tags_cols   = [c for c in df_y.columns if c.startswith("tags:")]
cat_cols    = [c for c in df_y.columns if c.startswith("categories:")]

In [6]:
df_y = df_y[genres_cols +cat_cols]

In [7]:
df_x.select_dtypes( 'string')


df_x_t = pd.get_dummies(
    df_x,
    columns=['release_month','price_status'],
    prefix=['m_','status_']
)

In [8]:
import torch
df_x_t = df_x_t.astype('float32')
X_t = torch.from_numpy(df_x_t.to_numpy())
print(X_t.shape, X_t.dtype)   # torch.Size([139496, 414]) torch.float32
df_y_t = df_y.astype('float32')
Y_t = torch.from_numpy(df_y_t.to_numpy())
print(Y_t.shape, Y_t.dtype)

torch.Size([139496, 414]) torch.float32
torch.Size([139496, 130]) torch.float32


C:\Users\Moha\AppData\Local\Temp\ipykernel_28252\1528426901.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  Y_t = torch.from_numpy(df_y_t.to_numpy())


In [9]:
import numpy as np 
import torch 
import torch.nn as nn 
import matplotlib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import torchmetrics
from torchmetrics.aggregation import MeanMetric



In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu") 
torch.manual_seed(42)
np.random.seed(42)
std = StandardScaler()


In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(X_t, Y_t, test_size=0.2, random_state=42)
X_test, X_val, Y_test, Y_val = train_test_split(X_test,Y_test,test_size=0.5,random_state= 42)


In [12]:
X_train,X_test, X_val = std.fit_transform(X_train), std.transform(X_test), std.transform(X_val)

In [24]:
class GenCatMLP(nn.Module):
    def __init__(self,in_dim,out_dim,hidden = 512,p=0.1):
        super().__init__()
        def block(ins,outs):
            return nn.Sequential(
                nn.Linear(ins,outs),
                nn.ReLU(),
                nn.Dropout(p),
                

            )
        self.net = nn.Sequential(block(in_dim,hidden),block(hidden,hidden//2),block(hidden//2,hidden//4),nn.Linear(hidden//4, out_dim))

    def forward(self,x):
        return self.net(x)
        


In [25]:
model = GenCatMLP(X_train.shape[1],Y_train.shape[1]).to(device)

In [26]:
loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.AdamW(params=model.parameters(),lr=0.00001,weight_decay=1e-4)

In [32]:
pos_counts = Y_train.sum(dim=0)              # [n_labels] positives per label
neg_counts = len(Y_train) - pos_counts        # [n_labels] negatives per label

# guard against zero positives (avoid inf / nan)
pos_counts = pos_counts.clamp(min=1)

raw_weight = neg_counts / pos_counts           # natural inverse-freq
pos_weight = raw_weight.clamp(max=20)           # cap at 5

pos_weight = pos_weight.to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [33]:
X_train = torch.from_numpy(X_train).float()
X_val   = torch.from_numpy(X_val).float()
X_test  = torch.from_numpy(X_test).float()

TypeError: expected np.ndarray (got Tensor)

In [34]:
X_train_ds = TensorDataset(X_train,Y_train)
train_loader = DataLoader(X_train_ds, batch_size= 256 , shuffle= True, num_workers = 0)
val_loader = DataLoader(TensorDataset(X_val,Y_val),batch_size= 512, shuffle= False )
test_loader = DataLoader(TensorDataset(X_test,Y_test), batch_size= 512 )
Dataiterator = iter(train_loader)
data = next(Dataiterator)

In [35]:
features, labels = data 
print(features.shape, labels.shape)

torch.Size([256, 414]) torch.Size([256, 130])


In [36]:
epochs = 100
epochs_ = [range(1,epochs)]
train_losses = []
val_losses = []
Patience = 5
epsilon = 1
F1macro = []
train_loss_metric = torchmetrics.aggregation.MeanMetric().to(device)
val_F1m_metric = torchmetrics.classification.MultilabelF1Score(Y_val.shape[1], average='macro').to(device)
val_loss_metric = torchmetrics.aggregation.MeanMetric().to(device)
for epoch in range(epochs):
    model.train()
    train_loss_metric.reset()
    for xb,yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optim.zero_grad()
        loss = loss_fn(model(xb),yb)
        train_loss_metric.update(loss.item(), weight = xb.size(0))
        loss.backward()
        optim.step()
    train_losses.append(train_loss_metric.compute().item())
    model.eval()
    val_F1m_metric.reset()
    val_loss_metric.reset()
    with torch.inference_mode():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            val_loss_metric.update(loss_fn(logits,yb).item(), weight = xb.size(0))
            val_F1m_metric.update(logits,yb)

        F1macro.append(val_F1m_metric.compute().item())
        val_losses.append(val_loss_metric.compute().item())

    if epoch % 10 == 0 :
        print(f'Training loss = {train_losses[-1]} validation loss is {val_losses[-1]}, F1macro = {F1macro[-1]}')

    
        





Training loss = 0.3630182445049286 validation loss is 0.3760640025138855, F1macro = 0.15130192041397095
Training loss = 0.35045504570007324 validation loss is 0.37601253390312195, F1macro = 0.1516677290201187
Training loss = 0.346344530582428 validation loss is 0.37613797187805176, F1macro = 0.15155129134655
Training loss = 0.34261441230773926 validation loss is 0.37689658999443054, F1macro = 0.15217356383800507
Training loss = 0.3386976718902588 validation loss is 0.37899044156074524, F1macro = 0.15306922793388367
Training loss = 0.33561623096466064 validation loss is 0.3791670799255371, F1macro = 0.1545872539281845
Training loss = 0.3324023187160492 validation loss is 0.3802850544452667, F1macro = 0.1545848250389099
Training loss = 0.32951289415359497 validation loss is 0.3823350965976715, F1macro = 0.15508215129375458
Training loss = 0.32685282826423645 validation loss is 0.38368600606918335, F1macro = 0.1601598560810089
Training loss = 0.3238633871078491 validation loss is 0.385535